In [1]:
## Customer-Churn-Prediction

import pandas as pd
from sklearn.preprocessing import LabelEncoder,StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split,RandomizedSearchCV,KFold,cross_val_score
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier


Train_Data=pd.read_excel("C:/Users/Lenovo/OneDrive/Documents/Churn_Train_Data.xlsx")
Train_Data=Train_Data.drop(columns=['customerID'])
Train_Data['Churn']=LabelEncoder().fit_transform(Train_Data['Churn'])
# Convert TotalCharges to numeric, replacing any non-numeric values with NaN
Train_Data['TotalCharges']=pd.to_numeric(Train_Data['TotalCharges'], errors='coerce')
X=Train_Data.drop(columns='Churn')
y=Train_Data['Churn']

## Dividing the features according as Numerics,Categories
num_cols=['tenure','MonthlyCharges','TotalCharges']
cat1_cols=['gender','Partner','Dependents','PhoneService','PaperlessBilling']
cat2_cols=['MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies','Contract','PaymentMethod']

## Building the Numerical Pipeline
num_pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='mean')),
    ('scaler',StandardScaler())
])

## Building the OrdinalEncoder Pipeline
cat1_pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1))
])

## Building the OneHotEncoder Pipeline
cat2_pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OneHotEncoder(handle_unknown='ignore'))
])

## Combining all the pipelines
preprocessor=ColumnTransformer([
    ('num',num_pipeline,num_cols),
    ('cat1',cat1_pipeline,cat1_cols),
    ('cat2',cat2_pipeline,cat2_cols)
])

## Train-Test-Split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

## Model Building
model=XGBClassifier(
    random_state=42
)
param_dist = {
    'n_estimators':[100,200,300],
    'max_depth':[1,3,5],
    'learning_rate': [0.1, 0.2]
}
kfold=KFold(n_splits=5,shuffle=True,random_state=42)
random_search = RandomizedSearchCV(
    model,
    param_dist,
    n_iter=10,
    cv=kfold
)

## Building Main Pipeline
pipeline=Pipeline([
    ('preprocessor',preprocessor),
    ('model',random_search)
])

pipeline.fit(X_train,y_train)
y_pred=pipeline.predict(X_test)
from sklearn.metrics import accuracy_score
accuracy=accuracy_score(y_test,y_pred)
print(accuracy)
scores=cross_val_score(pipeline,X,y,cv=kfold,scoring='accuracy')
print(scores)
print('Average accuracy :',scores.mean())


0.8075
[0.80375    0.80125    0.79875    0.805      0.79599499]
Average accuracy : 0.8009489987484356
